# Model Architecture
1. **Requirements to run this notebook**:
   1. Creating folder structure with the drive
   2. Mounting the google drive
   3. Storing and accessing OPENAI_API_KEY and PINECONE_API_KEY
2. **Dependecies**
3. **Data Loading**:
   1. Eval dataset reading and loading in dataframe
   2. Data Loading - Konwledge base data reading and loading into documents
4.  **Methods**:
    1. creating vector db indices(pinecone)
    2. creating embeddings and storing the vector db
    3. create an LLM chain for synthesis given a retriever
    4. Model infernce
    5. Evaluation with RAGAS
5. **RAG inference and evaluation**:
    - created a pipeline by stacking Methods in 4
    - pipleline is in the order of the methods
6. **Ground truth generation** for the validation dataset:
    - validation dataset doesn't have ground truth. So I used GPT-4o for ground truth generation with chunks having lots of overlap and having high number of retrieved documents to generate ground truth answers.
7. **Experiments**:
   - In total 6 experiments:
      1. Experiment 1
      2. Experiment 2
      3. Experiment 3
      4. Experiment 4
      5. Experiment 5
      6. Experiment 6
   - Each experiment have text blocks over them - that contain configurations of the experiment
   - configurations includes:
      - chunking methods
      -  embedding models
      - retrieval search type
      - retrieval candidates
8. **Comparative Analysis**
9. **Observations**: observations based on comparative analysis
10. **Conclusions**



# **1.1 Requirements to run the below code:**
1. You need a OpenAI API key and PineCone API Key to execute below code blocks
2. create a folder tifin:
    - create a folder pdf_docs within tifin folder
    - keep the pdf(Investment Case For Disruptive Innovation.pdf) in this pdf_docs. so the path of the pdf = /content/drive/MyDrive/tifin/pdf_docs
    - Store Evaluation_Questions.txt in tifin directory



# **1.2 - Mounting the google drive for storage access**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# **1.3 - Get the API secret keys for accessing OpenAI and Pinecone endpoints**

In [ ]:
import os
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['PINECONE_API_KEY'] = userdata.get('PINECONE_API_KEY')

# **2 - Dependencies:**

In [ ]:
! pip install -U -q langchain_community langchain-openai langchain langchain_experimental pypdf pinecone langchain-pinecone ragas datasets rank_bm25

In [ ]:
from tqdm import tqdm

from langchain.document_loaders.pdf import PyPDFDirectoryLoader

from langchain.prompts import PromptTemplate
from langchain_openai.chat_models import ChatOpenAI
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser
from langchain.schema.runnable import RunnableLambda, RunnablePassthrough

from pinecone import Pinecone as Pinecone_api
from pinecone import ServerlessSpec
from langchain_pinecone import PineconeVectorStore

from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker
from langchain.schema.document import Document

import pandas as pd

from ragas.metrics import (
    answer_relevancy,
    faithfulness,
    context_recall,
    context_precision,
    answer_correctness,
    answer_similarity
)

from ragas import evaluate

from datasets import Dataset

/usr/local/lib/python3.10/dist-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_name" in _VertexAIBase has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_name" in _VertexAICommon has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/ragas/metrics/__init__.py:4: LangChainDeprecationWarning: As of langchain-core 0.3.0, LangChain uses pydantic v2 internally. The langchain_core.pydantic_v1 module was a compatibility shim for pydantic v1, and should no longer be used. Please update the code to import from Pydantic directly.

For example, replace imports like: `from langchain_core.pydantic_v1 import BaseModel`
with: `

# **3.1 - Data Loading: Load the evaluation dataset questions:**
1. load the questions and store them in a pandas dataframe
2. required for benchmarking downstream

In [ ]:
query_path = '/content/drive/MyDrive/tifin/Evaluation_Questions.txt'

# Evaluation File loading
file1 = open(query_path, 'r')
Lines = file1.readlines()


eval_q_list = list()

for line in Lines:
  if not line.strip():
    continue
  eval_q_list.append({"question": line.strip()})

df_questions = pd.DataFrame(eval_q_list)


# **3.2 - Document Loader - PDF reader:**
1. Reads all the pages in the pdf


In [ ]:
# from pypdf import PdfReader
def load_documents(pdf_path):
  document_loader = PyPDFDirectoryLoader(pdf_path)
  return document_loader.load()

In [ ]:
pdf_path = '/content/drive/MyDrive/tifin/pdf_docs'
docs = load_documents(pdf_path)
print("Number of pages in the pdf:", len(docs))
print("One of the pages in the document is:", docs[0])

Number of pages in the pdf: 22
One of the pages in the document is: page_content='•
1Why Invest In Disruptive Innovation?
Sources: ARK Investment Management LLC, 2024. Forecasts are inherently limited and cannot be relied upon. For informational purposes only and should not be considered investment advice or a recommendation to buy, sell, or hold any particular security. Past performance is not indicative of future results.As of June 30, 2024' metadata={'source': '/content/drive/MyDrive/tifin/pdf_docs/Investment Case For Disruptive Innovation.pdf', 'page': 0}


# **4.1 - Methods: Initialize Vectordb store:**

In [ ]:
pc = Pinecone_api()

def create_pinecone_index(index_name, index_dimensions, metric, cloud, region):
  """
  method to create pinecone index using index params: name, dims, metric, cloud, region
  """
  index_name = index_name

  if index_name in pc.list_indexes().names():
      pc.delete_index(index_name)

  # # wait for index to be initialized

  pc.create_index(name=index_name,
                  dimension=index_dimensions, # Replace with your model dimensions
                  metric=metric, # Replace with your model metric
                  spec=ServerlessSpec(cloud=cloud, region=region))
  while not pc.describe_index(index_name).status['ready']:
      time.sleep(1)

def delete_pinecone_index(index_name):
  """
  Deletes pinecone existing indices
  As pinecone allows only 5 maximum serverless indices we need deletion option
  """
  if index_name in pc.list_indexes().names():
    pc.delete_index(index_name)


# **4.2 - Methods: Create embeddings for the chunks of Docs and store in vectordb:**
1. vector db based on differnt chunking strategies

In [ ]:
def create_vector_db(chunks, embeddings, index_name):
  vector_db = PineconeVectorStore.from_documents(chunks, embeddings, index_name=index_name)
  return vector_db

# **4.3 - Methods: Create an LLM chain:**

In [ ]:
def create_model_chain(model_name, temperature, template, retriever):

  chat_model = ChatOpenAI(model_name=model_name, temperature=temperature)
  prompt = PromptTemplate.from_template(template)
  chain = (
    {
        "context": itemgetter("question")|retriever,
        "question": itemgetter("question")
    }
    | RunnablePassthrough.assign(context=itemgetter("context"))
    |{"response": prompt|chat_model|StrOutputParser(), "context": itemgetter("context")}
    )
  return chain

# **4.4 - Methods: Model inference:**

In [ ]:
def answer_the_question(chain, query):
  response = chain.invoke({"question": query})
  return response

In [ ]:
# model answers
def model_answers_dataset(chain, df):
  rag_answer_synthesis_dataset = []
  for idx, row in df.iterrows():
    answer = answer_the_question(chain, query=row["question"])
    rag_answer_synthesis_dataset.append({
        "question": row["question"],
        "answer": answer["response"],
        "contexts": [context.page_content for context in answer["context"]],
        "ground_truth": row["ground_truth"]
    })
  df_ans = pd.DataFrame(rag_answer_synthesis_dataset)
  return df_ans
#         "reference": answer["context"][0].metadata['page'],

# **4.5 - Methods: Model evaluation using RAGAS:**

In [ ]:
def evaluate_rag_chain_with_ragas(ragas_dataset):
  result = evaluate(
    ragas_dataset,
    metrics=[
        context_precision,
        faithfulness,
        answer_relevancy,
        context_recall,
        answer_correctness,
        answer_similarity
    ],
  )
  return result
# "retrieved_contexts": [context.page_content for context in answer["context"]],

# **5 - RAG inference and eval pipeline**:

In [ ]:
def rag_inference_and_eval(index_name, index_dimensions, index_metric, index_cloud, index_region,
      embedding_model_name, chunks, k, search_type, df_eval_dataset, answer_synthesis_model_name,
      answer_synthesis_model_temperature, answer_synthesis_model_template):
  # index creation
  create_pinecone_index(index_name, index_dimensions, index_metric, index_cloud, index_region)
  # initializing the embeddings
  embeddings = OpenAIEmbeddings(model=embedding_model_name)
  # creating embeddings and storing in the vector db(pinecone)
  vector_db = create_vector_db(chunks, embeddings, index_name)
  # retriever
  vector_db_retriever = vector_db.as_retriever(search_type='similarity', search_kwargs={"k": k})
  # model chain
  chain = create_model_chain(answer_synthesis_model_name, answer_synthesis_model_temperature,
                             answer_synthesis_model_template, vector_db_retriever)
  # model output generation
  df_rag_dataset = model_answers_dataset(chain, df_eval_dataset)
  # converting into Dataset format
  rag_dataset = Dataset.from_pandas(df_rag_dataset)
  # evaluating using ragas
  metrics = evaluate_rag_chain_with_ragas(rag_dataset)
  return metrics

# **6 - Ground truth generation**:
1. Using LLM(GPT-4o) for getting ground truths given a retriever for context

In [ ]:
def get_ground_truth_datset(df):
  """
  method to create a ground truth dataset for evaluation given a retriever
  we use temperature 0 for groundtruth answer generation
  model - GPT-4o

  ground truth dataset generation has its own retriever
  """

  # retriver creation
  chunk_size = 600
  chunk_overlap = 450
  length_function = len
  is_separator_regex = False
  rcs_text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap,
                                                   length_function=length_function, is_separator_regex=is_separator_regex)

  chunks = rcs_text_splitter.split_documents(docs)
  # creating an index for simple text splitter chunking
  index_name = "pinecone-vdb-ground-truth"
  index_dimensions = 3072
  metric="cosine"
  cloud="aws"
  region="us-east-1"
  create_pinecone_index(index_name, index_dimensions, metric, cloud, region)

  # initialize embedding model
  embedding_model_name="text-embedding-3-large"
  embeddings = OpenAIEmbeddings(model=embedding_model_name)

  # vector_db name based recursive_character_splitter
  vector_db = create_vector_db(chunks, embeddings, index_name)

  # creating the retriever
  retriever = vector_db.as_retriever(search_kwargs={"k": 10})

  ground_truth_qa_model = ChatOpenAI(model='gpt-4o', temperature=0)

  ground_truth_template = """
  You are an expert in finance sector with great experience and knowledge in stock market.
  With your expertise answer the Question below based on the Context provided
  Context: {context}
  Question: {question}
  """

  prompt = PromptTemplate.from_template(ground_truth_template)

  ground_truth_qa_model_chain = (
      {
          "context": itemgetter("question")| retriever,
          "question": itemgetter("question")
      }
      | RunnablePassthrough.assign(context=itemgetter("context"))
      |{"response": prompt|ground_truth_qa_model|StrOutputParser(), "context": itemgetter("context")}
  )

  rag_eval_dataset = []
  for idx, row in df.iterrows():
    ground_truth = ground_truth_qa_model_chain.invoke({"question": row['question']})

    rag_eval_dataset.append({
        "question": row["question"],
        "ground_truth": ground_truth["response"]
    })

  df_ground_truth_dataset = pd.DataFrame(rag_eval_dataset)
  return df_ground_truth_dataset

In [ ]:
df_eval_dataset = get_ground_truth_datset(df_questions)

In [ ]:
df_eval_dataset.to_csv("/content/drive/MyDrive/tifin/ground_truth/ground_truth.csv")

In [ ]:
df_eval_dataset.head(5)

,question,ground_truth
0,Q01. What is the core objective of investing i...,The core objective of investing in disruptive ...
1,Q02. What are the significant risks associated...,"Based on the provided context, ARK highlights ..."
2,Q03. Can you list the converging innovation pl...,"Based on the provided context, ARK identifies ..."
3,Q04. How does ARK describe the impact of Artif...,ARK describes the impact of Artificial Intelli...
4,Q05. What transformative potential does Multio...,"According to ARK, Multiomic Sequencing holds s..."


In [ ]:
# delete pinecone ground truth retriever index as it is not necessary anymore
delete_pinecone_index(index_name = "pinecone-vdb-ground-truth")

In [ ]:
experiment_metric_collector = list()

# **7.1 - Experiment 1: Baseline**:
1. Experiment **configuration**:
     - Simple Text Splitting(STS) - (i.e. chunk overlap = 0)
     - embedding model - text-embedding-ada-002,
     - retrieval_search_type - similarity
     - retrieval candidates - k=5
2. Chunking: Recursive Character Splitter with chunk_over_lap = 0
3. create a pinecone index based on chunking strategy - simple Text Splitting
4. store the sts chunks embeddings into this pinecone index
5. create a retriever:
     - create a pincone index for the embedding model -  text-embedding-ada-002
6. get ground truths given a retriever
7. get model outputs
8. evaluate using RAGAS

In [ ]:
chunk_size = 300
chunk_overlap = 0
length_function = len
is_separator_regex = False
sts_text_splitter = RecursiveCharacterTextSplitter( chunk_size=chunk_size, chunk_overlap=chunk_overlap,
                                                   length_function=length_function, is_separator_regex=is_separator_regex)
chunks = sts_text_splitter.split_documents(docs)
print("number of chunks:", len(chunks))
print("example chunk:", chunks[0])

number of chunks: 157
example chunk: page_content='•
1Why Invest In Disruptive Innovation?' metadata={'source': '/content/drive/MyDrive/tifin/pdf_docs/Investment Case For Disruptive Innovation.pdf', 'page': 0}


In [ ]:
index_name = "pinecone-vdb-experiment1"
index_dimensions = 1536
index_metric = "cosine"
index_cloud = "aws"
index_region="us-east-1"
embedding_model_name="text-embedding-ada-002"
#chunks: defined above
k = 5
search_type = 'similarity'
#df_eval_dataset: defined above
answer_synthesis_model_name = 'gpt-3.5-turbo'
answer_synthesis_model_temperature = 0.5
answer_synthesis_model_template = """
You are an expert in finance sector with great experience and knowledge in stock market.
With your expertise answer the Question below based on the Context provided
Context: {context}
Question: {question}
"""

exp1_result_metrics = \
rag_inference_and_eval(index_name, index_dimensions, index_metric, index_cloud,
                       index_region, embedding_model_name, chunks, k, search_type,
                       df_eval_dataset, answer_synthesis_model_name,
                       answer_synthesis_model_temperature, answer_synthesis_model_template)

Evaluating:   0%|          | 0/120 [00:00<?, ?it/s]

In [ ]:
exp1_result_metrics

{'context_precision': 0.6632, 'faithfulness': 0.6148, 'answer_relevancy': 0.9528, 'context_recall': 0.2956, 'answer_correctness': 0.5841, 'answer_similarity': 0.9564}

In [ ]:
curr_exp1_result_metrics = dict()
curr_exp1_result_metrics["experiment"] = "experiment1"
curr_exp1_result_metrics.update(dict(exp1_result_metrics))
experiment_metric_collector.append(curr_exp1_result_metrics)

In [ ]:
baseline_results = exp1_result_metrics.to_pandas()
# save the model output to the drive including the metrics
baseline_results.to_csv("/content/drive/MyDrive/tifin/outputs/experiment1(baseline)_output.csv")

In [ ]:
# delete pinecone index for experiment 1
delete_pinecone_index(index_name = index_name)

# **7.2 - Experiment 2**:

1. Experiment **configuration**:
     - Simple Text Splitting(STS) ( i.e. chunk-overlap=0)
     - embedding model - text-embedding-003-large,
     - retrieval_search_type - similarity
     - retrieval candidates - k=5
2. Chunking: Recursive Character Splitter with chunk_over_lap = 0
3. create a pinecone index based on chunking strategy -  simple Text Splitting
4. store the sts chunks embeddings into this pinecone index
5. create a retriever:
    - create a pincone index for the embedding model -  text-embedding-003-large
6. get ground truths given a retriever
7. get model outputs
8. evaluate using RAGAS

In [ ]:
chunk_size = 300
chunk_overlap = 0
length_function = len
is_separator_regex = False
sts_text_splitter = RecursiveCharacterTextSplitter( chunk_size=chunk_size, chunk_overlap=chunk_overlap,
                                                   length_function=length_function, is_separator_regex=is_separator_regex)
chunks = sts_text_splitter.split_documents(docs)
print("number of chunks:", len(chunks))
print("example chunk:", chunks[0])

number of chunks: 157
example chunk: page_content='•
1Why Invest In Disruptive Innovation?' metadata={'source': '/content/drive/MyDrive/tifin/pdf_docs/Investment Case For Disruptive Innovation.pdf', 'page': 0}


In [ ]:
index_name = "pinecone-vdb-experiment2"
index_dimensions = 3072
index_metric = "cosine"
index_cloud = "aws"
index_region="us-east-1"
embedding_model_name="text-embedding-3-large"
#chunks: defined above
k = 5
search_type = 'similarity'
#df_eval_dataset: defined above
answer_synthesis_model_name = 'gpt-3.5-turbo'
answer_synthesis_model_temperature = 0.5
answer_synthesis_model_template = """
You are an expert in finance sector with great experience and knowledge in stock market.
With your expertise answer the Question below based on the Context provided
Context: {context}
Question: {question}
"""

exp2_result_metrics = \
rag_inference_and_eval(index_name, index_dimensions, index_metric, index_cloud,
                       index_region, embedding_model_name, chunks, k, search_type,
                       df_eval_dataset, answer_synthesis_model_name,
                       answer_synthesis_model_temperature, answer_synthesis_model_template)

Evaluating:   0%|          | 0/120 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[89]: TimeoutError()


In [ ]:
exp2_result_metrics

{'context_precision': 0.7650, 'faithfulness': 0.6497, 'answer_relevancy': 0.9092, 'context_recall': 0.5518, 'answer_correctness': 0.6202, 'answer_similarity': 0.9570}

In [ ]:
curr_exp2_result_metrics = dict()
curr_exp2_result_metrics["experiment"] = "experiment2"
curr_exp2_result_metrics.update(dict(exp2_result_metrics))
experiment_metric_collector.append(curr_exp2_result_metrics)

In [ ]:
exp2_results = exp2_result_metrics.to_pandas()
# save the model output to the drive including the metrics
exp2_results.to_csv("/content/drive/MyDrive/tifin/outputs/experiment2_output.csv")

In [ ]:
# delete pinecone index for experiment 2
delete_pinecone_index(index_name = index_name)

 # **7.3 - Experiment 3**:

1. Experiment **Configuration**:
     - Recursive Character Splitter(RCS) - I experimented with chunk_over_lap = 1/2 * chunk_size
     - embedding model - text-embedding-003-large,
     - retrieval_search_type - similarity
     - retrieval candidates - k=10
2. Chunking: Recursive Character Splitter with chunk_over_lap = 1/2 of chunk size
3. create a pinecone index based on chunking strategy -  Recursive Character Splitting
4. store the rcs chunks embeddings into this pinecone index
5. create a retriever:
     - embedding model - text-embedding-003-large
6. get ground truths given a retriever
7. get model outputs
8. evaluate using RAGAS

In [ ]:
chunk_size = 600
chunk_overlap = 300
length_function = len
is_separator_regex = False
rcs_text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap,
                                                   length_function=length_function, is_separator_regex=is_separator_regex)

chunks = rcs_text_splitter.split_documents(docs)
print("number of chunks:", len(chunks))
print("example chunk:", chunks[0])

number of chunks: 116
example chunk: page_content='•
1Why Invest In Disruptive Innovation?
Sources: ARK Investment Management LLC, 2024. Forecasts are inherently limited and cannot be relied upon. For informational purposes only and should not be considered investment advice or a recommendation to buy, sell, or hold any particular security. Past performance is not indicative of future results.As of June 30, 2024' metadata={'source': '/content/drive/MyDrive/tifin/pdf_docs/Investment Case For Disruptive Innovation.pdf', 'page': 0}


In [ ]:
index_name = "pinecone-vdb-experiment3"
index_dimensions = 3072
index_metric = "cosine"
index_cloud = "aws"
index_region="us-east-1"
embedding_model_name="text-embedding-3-large"
#chunks: defined above
k = 10
search_type = 'similarity'
#df_eval_dataset: defined above
answer_synthesis_model_name = 'gpt-3.5-turbo'
answer_synthesis_model_temperature = 0.5
answer_synthesis_model_template = """
You are an expert in finance sector with great experience and knowledge in stock market.
With your expertise answer the Question below based on the Context provided
Context: {context}
Question: {question}
"""

exp3_result_metrics = \
rag_inference_and_eval(index_name, index_dimensions, index_metric, index_cloud,
                       index_region, embedding_model_name, chunks, k, search_type,
                       df_eval_dataset, answer_synthesis_model_name,
                       answer_synthesis_model_temperature, answer_synthesis_model_template)

Evaluating:   0%|          | 0/120 [00:00<?, ?it/s]

In [ ]:
exp3_result_metrics

{'context_precision': 0.7584, 'faithfulness': 0.6599, 'answer_relevancy': 0.9471, 'context_recall': 0.8474, 'answer_correctness': 0.6067, 'answer_similarity': 0.9663}

In [ ]:
curr_exp3_result_metrics = dict()
curr_exp3_result_metrics["experiment"] = "experiment3"
curr_exp3_result_metrics.update(dict(exp3_result_metrics))
experiment_metric_collector.append(curr_exp3_result_metrics)

In [ ]:
exp3_results = exp3_result_metrics.to_pandas()
# save the model output to the drive including the metrics
exp3_results.to_csv("/content/drive/MyDrive/tifin/outputs/experiment3_output.csv")

In [ ]:
# delete pinecone index for experiment 3
delete_pinecone_index(index_name = index_name)

 # **7.4 - Experiment 4**:

1. Experiment **Configuration**:
     - Recursive Character Splitter(RCS) with Chunk_over_lap = 1/2 * chunk_size
     - ensemble retrival:
              * text-embedding-003-large as embedding model in one of the retrivers,
              * BM25Retriever
     - retrieval_search_type - similarity
     - retrieval candidates - k=10
2. Chunking: Recursive Character Splitter with chunk_over_lap = 1/2 of chunk size
3. create a pinecone index based on chunking strategy -  Recursive Character Splitting
4. store the rcs chunks embeddings into this pinecone index
5. create a retriever:
     - embedding model - text-embedding-003-large
6. get ground truths given a retriever
7. get model outputs
8. evaluate using RAGAS

In [ ]:
from langchain.retrievers import BM25Retriever, EnsembleRetriever

In [ ]:
chunk_size = 600
chunk_overlap = 300
length_function = len
is_separator_regex = False
rcs_text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap,
                                                   length_function=length_function, is_separator_regex=is_separator_regex)

chunks = rcs_text_splitter.split_documents(docs)
print("number of chunks:", len(chunks))
print("example chunk:", chunks[0])

number of chunks: 116
example chunk: page_content='•
1Why Invest In Disruptive Innovation?
Sources: ARK Investment Management LLC, 2024. Forecasts are inherently limited and cannot be relied upon. For informational purposes only and should not be considered investment advice or a recommendation to buy, sell, or hold any particular security. Past performance is not indicative of future results.As of June 30, 2024' metadata={'source': '/content/drive/MyDrive/tifin/pdf_docs/Investment Case For Disruptive Innovation.pdf', 'page': 0}


In [ ]:
index_name = "pinecone-vdb-experiment4"
index_dimensions = 3072
index_metric = "cosine"
index_cloud = "aws"
index_region="us-east-1"
embedding_model_name="text-embedding-3-large"
#chunks: defined above
k = 10
search_type = 'similarity'
#df_eval_dataset: defined above
answer_synthesis_model_name = 'gpt-3.5-turbo'
answer_synthesis_model_temperature = 0.5
answer_synthesis_model_template = """
You are an expert in finance sector with great experience and knowledge in stock market.
With your expertise answer the Question below based on the Context provided
Context: {context}
Question: {question}
"""


# index creation
create_pinecone_index(index_name, index_dimensions, index_metric, index_cloud, index_region)
# initializing the embeddings
embeddings = OpenAIEmbeddings(model=embedding_model_name)
# creating embeddings and storing in the vector db(pinecone)
vector_db_exp4 = create_vector_db(chunks, embeddings, index_name)
# retriever
bm25_retriever = BM25Retriever.from_documents(docs)
bm25_retriever.k = 2
vector_db_exp4_retriever = vector_db_exp4.as_retriever(search_type='similarity', search_kwargs={"k": k})
ensemble_retriever = EnsembleRetriever(retrievers=[bm25_retriever, vector_db_exp4_retriever], weights=[0.75, 0.25])
# model chain
chain = create_model_chain(answer_synthesis_model_name, answer_synthesis_model_temperature,
                            answer_synthesis_model_template, ensemble_retriever)
# model output generation
df_rag_dataset = model_answers_dataset(chain, df_eval_dataset)
# converting into Dataset format
rag_dataset = Dataset.from_pandas(df_rag_dataset)
# evaluating using ragas
exp4_result_metrics = evaluate_rag_chain_with_ragas(rag_dataset)


Evaluating:   0%|          | 0/120 [00:00<?, ?it/s]

In [ ]:
exp4_result_metrics

{'context_precision': 0.8063, 'faithfulness': 0.7385, 'answer_relevancy': 0.9542, 'context_recall': 0.8795, 'answer_correctness': 0.6946, 'answer_similarity': 0.9618}

In [ ]:
curr_exp4_result_metrics = dict()
curr_exp4_result_metrics["experiment"] = "experiment4"
curr_exp4_result_metrics.update(dict(exp4_result_metrics))
experiment_metric_collector.append(curr_exp4_result_metrics)

In [ ]:
exp4_results = exp4_result_metrics.to_pandas()
# save the model output to the drive including the metrics
exp4_results.to_csv("/content/drive/MyDrive/tifin/outputs/experiment4_output.csv")

In [ ]:
# delete pinecone index for experiment 4
delete_pinecone_index(index_name = index_name)

 #  **7.5 - Experiment 5**:

1. Experiment **Configuration**:
     - SemanticChunker for chunking - This is an experimantal chunking strategy by langchain(still in beta)
     - embedding- text-embedding-003-large
     - retrieval_search_type - similarity
     - retrieval candidates - k=10
2. Chunking: Semantic Chunking
3. create a pinecone index based on chunking strategy -  Recursive Character Splitting
4. store the rcs chunks embeddings into this pinecone index
5. create a retriever:
     - embedding model - text-embedding-003-large
6. get ground truths given a retriever
7. get model outputs
8. evaluate using RAGAS

In [ ]:
# initialize embedding model
embedding_model_name="text-embedding-3-large"
embeddings = OpenAIEmbeddings(model=embedding_model_name)

semantic_text_splitter = SemanticChunker(embeddings)

chunks = semantic_text_splitter.split_documents(docs)
print("number of chunks:", len(chunks))
print("example chunk:", chunks[0])

number of chunks: 45
example chunk: page_content='•
1Why Invest In Disruptive Innovation? Sources: ARK Investment Management LLC, 2024. Forecasts are inherently limited and cannot be relied upon.' metadata={'source': '/content/drive/MyDrive/tifin/pdf_docs/Investment Case For Disruptive Innovation.pdf', 'page': 0}


In [ ]:
index_name = "pinecone-vdb-experiment5"
index_dimensions = 3072
index_metric = "cosine"
index_cloud = "aws"
index_region="us-east-1"
embedding_model_name="text-embedding-3-large"
#chunks: defined above
k = 10
search_type = 'similarity'
#df_eval_dataset: defined above
answer_synthesis_model_name = 'gpt-3.5-turbo'
answer_synthesis_model_temperature = 0.5
answer_synthesis_model_template = """
You are an expert in finance sector with great experience and knowledge in stock market.
With your expertise answer the Question below based on the Context provided
Context: {context}
Question: {question}
"""

exp5_result_metrics = \
rag_inference_and_eval(index_name, index_dimensions, index_metric, index_cloud,
                       index_region, embedding_model_name, chunks, k, search_type,
                       df_eval_dataset, answer_synthesis_model_name,
                       answer_synthesis_model_temperature, answer_synthesis_model_template)

Evaluating:   0%|          | 0/120 [00:00<?, ?it/s]

In [ ]:
exp5_result_metrics

{'context_precision': 0.7068, 'faithfulness': 0.6791, 'answer_relevancy': 0.9444, 'context_recall': 0.7254, 'answer_correctness': 0.6332, 'answer_similarity': 0.9610}

In [ ]:
curr_exp5_result_metrics = dict()
curr_exp5_result_metrics["experiment"] = "experiment5"
curr_exp5_result_metrics.update(dict(exp5_result_metrics))
experiment_metric_collector.append(curr_exp5_result_metrics)

In [ ]:
exp5_results = exp5_result_metrics.to_pandas()
# save the model output to the drive including the metrics
exp5_results.to_csv("/content/drive/MyDrive/tifin/outputs/experiment5_output.csv")

In [ ]:
# delete pinecone index for experiment 5
delete_pinecone_index(index_name = index_name)

# **7.6 - Experiment 6**
1. Experiment **Configuration**:
     - Recursive Character Splitter(RCS) - I experimented with chunk_over_lap = 1/2 * chunk_size
     - embedding model - text-embedding-003-large,
     - retrieval_search_type - MMR
     - retrieval candidates - k=10
2. Chunking: Recursive Character Splitter with chunk_over_lap = 1/2 of chunk size
3. create a pinecone index based on chunking strategy -  Recursive Character Splitting
4. store the rcs chunks embeddings into this pinecone index
5. create a retriever:
     - embedding model - text-embedding-003-large
6. get ground truths given a retriever
7. get model outputs
8. evaluate using RAGAS

In [ ]:
chunk_size = 600
chunk_overlap = 300
length_function = len
is_separator_regex = False
rcs_text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap,
                                                   length_function=length_function, is_separator_regex=is_separator_regex)

chunks = rcs_text_splitter.split_documents(docs)
print("number of chunks:", len(chunks))
print("example chunk:", chunks[0])

number of chunks: 116
example chunk: page_content='•
1Why Invest In Disruptive Innovation?
Sources: ARK Investment Management LLC, 2024. Forecasts are inherently limited and cannot be relied upon. For informational purposes only and should not be considered investment advice or a recommendation to buy, sell, or hold any particular security. Past performance is not indicative of future results.As of June 30, 2024' metadata={'source': '/content/drive/MyDrive/tifin/pdf_docs/Investment Case For Disruptive Innovation.pdf', 'page': 0}


In [ ]:
index_name = "pinecone-vdb-experiment6"
index_dimensions = 3072
index_metric = "cosine"
index_cloud = "aws"
index_region="us-east-1"
embedding_model_name="text-embedding-3-large"
#chunks: defined above
k = 10
search_type = 'mmr'
#df_eval_dataset: defined above
answer_synthesis_model_name = 'gpt-3.5-turbo'
answer_synthesis_model_temperature = 0.5
answer_synthesis_model_template = """
You are an expert in finance sector with great experience and knowledge in stock market.
With your expertise answer the Question below based on the Context provided
Context: {context}
Question: {question}
"""

exp6_result_metrics = \
rag_inference_and_eval(index_name, index_dimensions, index_metric, index_cloud,
                       index_region, embedding_model_name, chunks, k, search_type,
                       df_eval_dataset, answer_synthesis_model_name,
                       answer_synthesis_model_temperature, answer_synthesis_model_template)

Evaluating:   0%|          | 0/120 [00:00<?, ?it/s]

In [ ]:
exp6_result_metrics

{'context_precision': 0.7414, 'faithfulness': 0.7459, 'answer_relevancy': 0.9508, 'context_recall': 0.8307, 'answer_correctness': 0.6205, 'answer_similarity': 0.9639}

In [ ]:
curr_exp6_result_metrics = dict()
curr_exp6_result_metrics["experiment"] = "experiment6"
curr_exp6_result_metrics.update(dict(exp6_result_metrics))
experiment_metric_collector.append(curr_exp6_result_metrics)

In [ ]:
exp6_results = exp6_result_metrics.to_pandas()
# save the model output to the drive including the metrics
exp6_results.to_csv("/content/drive/MyDrive/tifin/outputs/experiment6_output.csv")

In [ ]:
# delete pinecone index for experiment 6
delete_pinecone_index(index_name = index_name)

# **8 - Comparitive Analysis**

In [ ]:
df_metrics = pd.DataFrame(experiment_metric_collector)

In [ ]:
df_metrics

,experiment,context_precision,faithfulness,answer_relevancy,context_recall,answer_correctness,answer_similarity
0,experiment1,0.663194,0.614758,0.952779,0.295575,0.584100,0.956399
1,experiment2,0.765000,0.649742,0.909175,0.551768,0.620206,0.957026
2,experiment3,0.758417,0.659932,0.947092,0.847381,0.606721,0.966310
3,experiment4,0.806282,0.738461,0.954209,0.879524,0.694577,0.961805
4,experiment5,0.706759,0.679127,0.944356,0.725385,0.633158,0.960987
5,experiment6,0.741402,0.745917,0.950824,0.830714,0.620507,0.963867


In [ ]:
df_metrics.to_csv("/content/drive/MyDrive/tifin/comparative_analysis.csv")

# **9- Observations**:

**Metrics meaning:**
1. Context precision: Measures the accuracy of retrieved documents.
2. Faithfulness: Assesses how well the generated answer adheres to the retrieved context.
3. Answer relevancy: Evaluates if the generated answer is relevant to the query.4. Context recall: Measures the completeness of the retrieved documents.
5. Answer correctness: Assesses the factual accuracy of the generated answer.
6. Answer similarity: Measures the similarity between the generated answer and the gold standard(expected answer).

**In general the RAG systems is performing well across experiment** because of:
- High answer relevancy: model is effectively retrieving and using relevant information to the asked query.
- Good answer similarity: generated answers are aligning well with the expected responses.
- Consistent performance across experiments: The metrics are relatively stable across the different experiments, suggesting a robust model.


**Best RAG Performer** based on the above experiments:

1. Context precision: Experiment 3 has the highest context precision, implying it retrieves the most accurate documents.
2. Faithfulness: Experiment 3 has the highest faithfulness score, as it generates answers that are most closely aligned with the retrieved context.
3. Answer relevancy: Experiment 3 also has the highest answer relevancy as generated answers are most relevant to the query.
4. Experiment 4 has highest context recall implying that it retrieves the most comprehensive set of documents. Better in the tasks of answering in generically to questions

  Across experiment - Experiment 3 has the best RAG performance.



**Best Overall System Performance**:
To evaluate the overall performance we need to consider both the retrival and generation metrics and answer corectness:
1. Experiment 3 has high scores in:
     - context precision
     - faithfulness, and
     - answer relevancy
     - context recall
     - answer corectness
     - answer similarity
2. Experiment 3 has great RAG-specific metrics,
4. Experiment 3 has great overall system performance in answer synthesis -  answer correctness.
5. Answer similarity: All experiments have relatively high answer similarity scores, indicating that the generated answers are consistently aligned with the gold standard.

So overall experiment 3 is a top performer


# **10 - Conclusion:**
The choice of experiment depends on the specific functionality of application.
But based on our validation for answering questions on Investment insights into ARK ETF fund -  Experiment 3 is best candidate.